## Setup

### Configuration

| Parameter | Value |
|-----------|-------|
| Apache Spark version | 4.0.2 |
| Apache Iceberg version | 1.10.0 |
| Number of Executors | 2 |
| Executor memory | 8GB |
| Cores per executor | 2 |
| Data size | 30GB |

In [1]:
# Stop any existing session
try:
    spark.stop()
except:
    pass

26/05/09 10:39:19 ERROR TransportRequestHandler: Error sending result StreamResponse[streamId=/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar,byteCount=47073384,body=FileSegmentManagedBuffer[file=/opt/spark/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar,offset=0,length=47073384]] to /172.18.0.9:56808; closing connection
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel.close(ChannelPromise)(Unknown Source)
26/05/09 10:39:19 ERROR TransportRequestHandler: Error sending result StreamResponse[streamId=/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar,byteCount=47073384,body=FileSegmentManagedBuffer[file=/opt/spark/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar,offset=0,length=47073384]] to /172.18.0.8:57586; closing connection
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel.close(ChannelPromise)(Unknown Source)


In [2]:
workshop_name = "file-size-deep-dive"

from pyspark.sql import SparkSession

executor_memory = "16g"
executor_cores = 2
num_executors = 2


spark = SparkSession.builder \
        .appName(workshop_name) \
        .master("spark://spark-master:7077") \
        .config("spark.executor.memory", executor_memory) \
        .config("spark.executor.cores", executor_cores) \
        .config("spark.executor.instances", num_executors) \
        .config("spark.cores.max", executor_cores * num_executors) \
    .getOrCreate()

Setting Spark log level to "WARN".


* Open [http://localhost:4040](http://localhost:4040) to see your Spark UI

In [ ]:
# Create Fake Data 
import os 
import glob
from pathlib import Path

from generate_data import run
from run_ddl import run_ddl 

################################################################################
## CHANGE DATA SIZE AS NEEDED
################################################################################

data_size_gb = 30

################################################################################
## CHANGE overwrite_existing_data TO True TO CREATE DATA
################################################################################
overwrite_existing_data = False 
data_folder = './data'

# Only when overwrite = False and there are atleast 1 csv files we need to skip data gen (everyother time we need to generate data) 
if not (not overwrite_existing_data and len(glob.glob(os.path.join(data_folder, '*.csv'))) > 1):
    print(f'Creating raw TPCH data in {data_folder}')
    run(scaling_factor=data_size_gb)

print(f'Loading data in raw TPCH folder {data_folder} to Iceberg Tables')
run_ddl(data_path=Path(data_folder), spark=spark, recreate=True) # Load created data into Iceberg tables on Minio (S3 equivalent)

In [3]:
%%sql
use prod.db

26/05/09 10:39:20 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


++
||
++
++

## Many small files => Spark wastes time opening file

In [ ]:
%%sql
SELECT
  MONTH (l_receiptdate) AS receipt_month,
  COUNT(*) AS num_line_items
FROM lineitem
GROUP BY 1 ORDER BY 2 desc LIMIT 10

## Maintenance functions are the simplest fix

In [ ]:
%%sql
-- new demo table
CREATE TABLE prod.db.lineitem_resized AS SELECT * FROM prod.db.lineitem;

In [ ]:
# call maintenance function on the newly created table
spark.sql("CALL demo.system.rewrite_data_files('prod.db.lineitem_resized')")

In [ ]:
%%sql
SELECT
  MONTH (l_receiptdate) AS receipt_month,
  COUNT(*) AS num_line_items
FROM lineitem_resized
GROUP BY 1 ORDER BY 2 desc LIMIT 10

## Partition data before insert

In [29]:
%%sql
CREATE TABLE
  IF NOT EXISTS prod.db.lineitem_part_year (
    l_orderkey BIGINT,
    l_partkey BIGINT,
    l_suppkey BIGINT,
    l_linenumber INT,
    l_quantity DECIMAL(15, 2),
    l_extendedprice DECIMAL(15, 2),
    l_discount DECIMAL(15, 2),
    l_tax DECIMAL(15, 2),
    l_returnflag STRING,
    l_linestatus STRING,
    l_shipdate DATE,
    l_commitdate DATE,
    l_receiptdate DATE,
    l_shipinstruct STRING,
    l_shipmode STRING,
    l_comment STRING
  ) USING iceberg PARTITIONED BY (YEAR (l_shipdate)) TBLPROPERTIES (
    'format-version' = '2'
  );

++
||
++
++

In [30]:
%%sql
INSERT INTO prod.db.lineitem_part_year
SELECT * FROM prod.db.lineitem;

++
||
++
++

In [31]:
%%sql
SELECT
  ROW_NUMBER() OVER (ORDER BY file_path) AS row_num,
  file_path,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb,
  split(file_path, '-')[1] AS writer_task
FROM prod.db.lineitem_part_year.files
ORDER BY 2, 3

26/05/09 11:20:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/09 11:20:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/09 11:20:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/09 11:20:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/09 11:20:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


row_num,file_path,file_size_mb,writer_task
1,s3://warehouse/prod/db/lineitem_part_year/data/l_shipdate_year=1992/00018-1308-98b5e18b-4c00-4471-8e1d-9e12cab6c217-0-00001.parquet,101.67,1308
2,s3://warehouse/prod/db/lineitem_part_year/data/l_shipdate_year=1992/00019-1309-98b5e18b-4c00-4471-8e1d-9e12cab6c217-0-00001.parquet,101.55,1309
3,s3://warehouse/prod/db/lineitem_part_year/data/l_shipdate_year=1992/00020-1310-98b5e18b-4c00-4471-8e1d-9e12cab6c217-0-00001.parquet,101.79,1310
4,s3://warehouse/prod/db/lineitem_part_year/data/l_shipdate_year=1992/00021-1311-98b5e18b-4c00-4471-8e1d-9e12cab6c217-0-00001.parquet,100.78,1311
5,s3://warehouse/prod/db/lineitem_part_year/data/l_shipdate_year=1992/00022-1312-98b5e18b-4c00-4471-8e1d-9e12cab6c217-0-00001.parquet,100.91,1312
6,s3://warehouse/prod/db/lineitem_part_year/data/l_shipdate_year=1992/00023-1313-98b5e18b-4c00-4471-8e1d-9e12cab6c217-0-00001.parquet,43.58,1313
7,s3://warehouse/prod/db/lineitem_part_year/data/l_shipdate_year=1993/00024-1314-98b5e18b-4c00-4471-8e1d-9e12cab6c217-0-00001.parquet,100.44,1314
8,s3://warehouse/prod/db/lineitem_part_year/data/l_shipdate_year=1993/00025-1315-98b5e18b-4c00-4471-8e1d-9e12cab6c217-0-00001.parquet,100.04,1315
9,s3://warehouse/prod/db/lineitem_part_year/data/l_shipdate_year=1993/00026-1316-98b5e18b-4c00-4471-8e1d-9e12cab6c217-0-00001.parquet,100.04,1316
10,s3://warehouse/prod/db/lineitem_part_year/data/l_shipdate_year=1993/00027-1317-98b5e18b-4c00-4471-8e1d-9e12cab6c217-0-00001.parquet,99.73,1317


In [23]:
%%sql
CREATE TABLE
  IF NOT EXISTS prod.db.lineitem_part_year_2gb_intask_mem (
    l_orderkey BIGINT,
    l_partkey BIGINT,
    l_suppkey BIGINT,
    l_linenumber INT,
    l_quantity DECIMAL(15, 2),
    l_extendedprice DECIMAL(15, 2),
    l_discount DECIMAL(15, 2),
    l_tax DECIMAL(15, 2),
    l_returnflag STRING,
    l_linestatus STRING,
    l_shipdate DATE,
    l_commitdate DATE,
    l_receiptdate DATE,
    l_shipinstruct STRING,
    l_shipmode STRING,
    l_comment STRING
  ) USING iceberg PARTITIONED BY (YEAR (l_shipdate)) TBLPROPERTIES (
    'format-version' = '2',
    'write.spark.advisory-partition-size-bytes' = '2147483648' -- 2 GB
  );

++
||
++
++

In [25]:
%%sql
INSERT INTO prod.db.lineitem_part_year_2gb_intask_mem
SELECT * FROM prod.db.lineitem;

++
||
++
++

In [26]:
%%sql
SELECT
  ROW_NUMBER() OVER (ORDER BY file_path) AS row_num,
  file_path,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb,
  split(file_path, '-')[1] AS writer_task
FROM prod.db.lineitem_part_year_2gb_intask_mem.files
ORDER BY 2, 3

26/05/09 11:08:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/09 11:08:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/09 11:08:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/09 11:08:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/09 11:08:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


row_num,file_path,file_size_mb,writer_task
1,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1992/00004-912-a9b39f0f-0c58-4b83-99ab-88192e37e719-0-00001.parquet,510.84,912
2,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1992/00004-912-a9b39f0f-0c58-4b83-99ab-88192e37e719-0-00002.parquet,38.26,912
3,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1993/00005-913-a9b39f0f-0c58-4b83-99ab-88192e37e719-0-00001.parquet,510.63,913
4,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1993/00005-913-a9b39f0f-0c58-4b83-99ab-88192e37e719-0-00002.parquet,36.03,913
5,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1993/00006-914-a9b39f0f-0c58-4b83-99ab-88192e37e719-0-00001.parquet,113.78,914
6,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1994/00009-917-a9b39f0f-0c58-4b83-99ab-88192e37e719-0-00001.parquet,510.64,917
7,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1994/00009-917-a9b39f0f-0c58-4b83-99ab-88192e37e719-0-00002.parquet,35.88,917
8,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1994/00010-918-a9b39f0f-0c58-4b83-99ab-88192e37e719-0-00001.parquet,113.8,918
9,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1995/00007-915-a9b39f0f-0c58-4b83-99ab-88192e37e719-0-00001.parquet,510.94,915
10,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/l_shipdate_year=1995/00007-915-a9b39f0f-0c58-4b83-99ab-88192e37e719-0-00002.parquet,39.59,915


In [27]:
# This forces same-year rows to the same Icebreg write task
import pyspark.sql.functions as F

spark.table("prod.db.lineitem")\
  .repartition(F.year(F.col("l_shipdate"))) \
  .writeTo("prod.db.lineitem_part_year_2gb_intask_mem") \
  .createOrReplace()

In [28]:
%%sql
SELECT
  ROW_NUMBER() OVER (ORDER BY file_path) AS row_num,
  file_path,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb,
  split(file_path, '-')[1] AS writer_task
FROM prod.db.lineitem_part_year_2gb_intask_mem.files
ORDER BY 2, 3

26/05/09 11:14:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/09 11:14:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/09 11:14:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/09 11:14:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/09 11:14:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


row_num,file_path,file_size_mb,writer_task
1,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/00000-1101-eeb8db48-12f6-4cbf-b26c-d271c01f6a79-0-00001.parquet,512.6,1101
2,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/00000-1101-eeb8db48-12f6-4cbf-b26c-d271c01f6a79-0-00002.parquet,181.61,1101
3,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/00001-1102-eeb8db48-12f6-4cbf-b26c-d271c01f6a79-0-00001.parquet,512.52,1102
4,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/00001-1102-eeb8db48-12f6-4cbf-b26c-d271c01f6a79-0-00002.parquet,184.57,1102
5,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/00002-1103-eeb8db48-12f6-4cbf-b26c-d271c01f6a79-0-00001.parquet,512.63,1103
6,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/00002-1103-eeb8db48-12f6-4cbf-b26c-d271c01f6a79-0-00002.parquet,182.74,1103
7,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/00003-1104-eeb8db48-12f6-4cbf-b26c-d271c01f6a79-0-00001.parquet,512.31,1104
8,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/00003-1104-eeb8db48-12f6-4cbf-b26c-d271c01f6a79-0-00002.parquet,6.49,1104
9,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/00004-1105-eeb8db48-12f6-4cbf-b26c-d271c01f6a79-0-00001.parquet,512.22,1105
10,s3://warehouse/prod/db/lineitem_part_year_2gb_intask_mem/data/00004-1105-eeb8db48-12f6-4cbf-b26c-d271c01f6a79-0-00002.parquet,189.92,1105
